In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import sum, expr, col, when, lit, regexp_extract, percentile_approx, regexp_replace, expr, count, to_timestamp, unix_timestamp, mean
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
import pyspark.sql.types as T
from pyspark.sql.window import Window

from pyspark.storagelevel import StorageLevel
from graphframes import GraphFrame

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import holidays


Goal: Take otpw_60m and select a few features, add temporal features, add graph features, clean up data, impute data, and separate into train/val/test datasets

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

# Checkpointing Info
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

In [0]:
#df_baseline = spark.read.format("csv").option("header","true").load(f"/student-groups/Group_3_1/data_separated_otpw_60m/train")
#df_baseline = spark.read.parquet(f"/student-groups/Group_3_1/data_separated_otpw_60m/train")
# df_baseline = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M/")
df_baseline = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")

In [0]:
df_baseline.printSchema()

In [0]:
display(df_baseline.limit(10))

In [0]:
df_baseline.printSchema()

In [0]:
#df_unpacked = df_baseline.withColumn("features_array", vector_to_array("features"))

# Get number of features
#n_features = len(df_unpacked.select("features_array").first()[0])

# Create columns
#for i in range(n_features):
#    df_unpacked = df_unpacked.withColumn(f"feature_{i}", F.col("features_array")[i])
#df_unpacked = df_unpacked.drop("features_array")
#display(df_unpacked.limit(10))

In [0]:
df_baseline = df_baseline.withColumn(
    "CRS_DEP_TIME_PAD",
    F.lpad(F.col("CRS_DEP_TIME"), 4, "0")
)

df_baseline = df_baseline.withColumn(
    "CRS_DEP_TIME_FMT",
    F.concat_ws(
        ":",
        F.substring("CRS_DEP_TIME_PAD", 1, 2),
        F.substring("CRS_DEP_TIME_PAD", 3, 2)
    )
)

df_baseline = df_baseline.withColumn(
    "sched_dep_ts",
    F.to_timestamp(
        F.concat_ws(" ", F.col("FL_DATE"), F.col("CRS_DEP_TIME_FMT")),
        "yyyy-MM-dd HH:mm"
    )
)

df_baseline = df_baseline.withColumn(
    "flight_id",
    F.concat_ws("_",
        F.col("FL_DATE"),
        F.col("OP_UNIQUE_CARRIER"),
        F.col("OP_CARRIER_FL_NUM"),
        F.col("ORIGIN"),
        F.col("DEST"),
        F.col("CRS_DEP_TIME")
    )
)


df_baseline = df_baseline.withColumn("YEAR", F.year("FL_DATE")).withColumn("MONTH", F.month("FL_DATE"))

In [0]:
df_baseline.count()

In [0]:
select_cols = ["DEP_DEL15", "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_DEP_TIME_FMT", "sched_dep_ts",
               "DEP_DELAY", "ARR_DELAY", "ORIGIN", "DEST", "TAIL_NUM", "FL_DATE", "flight_id", 
               "OP_CARRIER_FL_NUM", "YEAR", "MONTH", 
               "origin_iata_code", "origin_station_lat", "origin_station_lon"]

In [0]:
df_baseline_filtered = df_baseline.select(select_cols)

In [0]:
df_baseline_filtered.printSchema()


## Create Temporal Features

In [0]:
# Parameters
LOOKBACK_1H = 3600 * 1
LOOKBACK_2H = 3600 * 2
LOOKBACK_3H = 3600 * 3
LOOKBACK_6H = 3600 * 6
CUTOFF_BEFORE_DEPARTURE = 3600 * 2 # 2 hours

In [0]:
def add_tail_turnaround_features(df):
    
    # Window for rolling 10h avg
    w_avg = (
        Window.partitionBy("TAIL_NUM")
              .orderBy(F.col("sched_dep_ts").cast("long"))
              .rangeBetween(-3600*12, -CUTOFF_BEFORE_DEPARTURE)  # aircraft history 10h
    )

    df = df.withColumn("tail_avg_delay_10h",
                       F.avg("arr_delay").over(w_avg))
    
    # Window for previous flight (row-based)
    w_lag = Window.partitionBy("TAIL_NUM").orderBy(F.col("sched_dep_ts").cast("long"))

    
    df = df.withColumn("tail_recent_delay",
                       F.lag("arr_delay", 1).over(w_lag))

    return df

In [0]:
df_baseline_filtered = add_tail_turnaround_features(df_baseline_filtered)

In [0]:
df_baseline_filtered.printSchema()

In [0]:
def add_route_history_features(df):
    
    w_route = (
        Window.partitionBy("ORIGIN", "DEST")
              .orderBy(F.col("sched_dep_ts").cast("long"))
              .rangeBetween(-3600*8, CUTOFF_BEFORE_DEPARTURE)
    )

    df = df.withColumn("route_avg_delay_6h",
                       F.avg("dep_delay").over(w_route))
    
    df = df.withColumn("route_max_delay_6h",
                       F.max("dep_delay").over(w_route))

    return df

In [0]:
df_baseline_filtered = add_route_history_features(df_baseline_filtered)

In [0]:
df_baseline_filtered.printSchema()

In [0]:
def add_airport_delay_windows(df):

  w = (
    Window.partitionBy("ORIGIN")
       .orderBy(F.col("sched_dep_ts").cast("long"))
       .rangeBetween(-LOOKBACK_6H - CUTOFF_BEFORE_DEPARTURE, -CUTOFF_BEFORE_DEPARTURE) # 6h max window
  )

  # base features
  df = df.withColumn("dep_delay_1h",
      F.avg("dep_delay").over(w.rangeBetween(-LOOKBACK_1H - CUTOFF_BEFORE_DEPARTURE, -CUTOFF_BEFORE_DEPARTURE)))
  df = df.withColumn("dep_delay_3h",
      F.avg("dep_delay").over(w.rangeBetween(-LOOKBACK_3H - CUTOFF_BEFORE_DEPARTURE, -CUTOFF_BEFORE_DEPARTURE)))
  df = df.withColumn("dep_delay_6h",
      F.avg("dep_delay").over(w.rangeBetween(-LOOKBACK_6H - CUTOFF_BEFORE_DEPARTURE, -CUTOFF_BEFORE_DEPARTURE)))

  # counts
  df = df.withColumn("num_flights_3h",
      F.count("flight_id").over(w.rangeBetween(-LOOKBACK_3H - CUTOFF_BEFORE_DEPARTURE, -CUTOFF_BEFORE_DEPARTURE)))

  # max delays
  df = df.withColumn("max_dep_delay_3h",
      F.max("dep_delay").over(w.rangeBetween(-LOOKBACK_6H - CUTOFF_BEFORE_DEPARTURE, -CUTOFF_BEFORE_DEPARTURE)))



  # daily sequence features
  window_day = Window.partitionBy("FL_DATE").orderBy("CRS_DEP_TIME")
  df = df.withColumn("departure_sequence", F.row_number().over(window_day))

  # Accumulated scheduled flights before current flight
  window_cum_flights = Window.partitionBy("ORIGIN", "FL_DATE").orderBy("CRS_DEP_TIME", "OP_CARRIER_FL_NUM").rowsBetween(Window.unboundedPreceding, -1)
  df = df.withColumn("scheduled_flights_before_current", F.count(F.lit(1)).over(window_cum_flights))
  df = df.fillna({"scheduled_flights_before_current": 0})

  # early morning delay propagation
  df = df.withColumn("dep_hour", (F.col("CRS_DEP_TIME") / 100).cast("int"))

  # Boolean → int is cheaper than boolean filters
  df = df.withColumn("is_early_morning", (F.col("dep_hour") < 8).cast("int"))

  # Compute morning delay rate for each airport per day
  morning_rates = (
    df.groupBy("ORIGIN", "FL_DATE")
    .agg(
        F.avg(
            F.when(F.col("is_early_morning") == 1, F.col("DEP_DEL15").cast("int"))
            .otherwise(None)
        ).alias("morning_delay_rate_airport_day")
    )
  )

  # Compute historical average orning delay per airport
  airport_avg_morning = (
    morning_rates.groupBy("ORIGIN")
    .agg(F.avg("morning_delay_rate_airport_day")
       .alias("avg_morning_delay_rate_airport"))
  )

  # Combine both
  morning_rates = (
    morning_rates.join(airport_avg_morning, on="ORIGIN", how="left")
  )

  # Join back to the main df
  df = df.join(
    morning_rates,
    on=["ORIGIN", "FL_DATE"],
    how="left"
  )

  # Add some derived features based on daily and historic morning delay rates
  df = (
    df.withColumn(
        "delta_morning_vs_hist",
        F.col("morning_delay_rate_airport_day") - F.col("avg_morning_delay_rate_airport")
    )
    .withColumn(
        "ratio_morning_vs_hist", F.when(F.col("avg_morning_delay_rate_airport") == 0, 0).otherwise(
        F.col("morning_delay_rate_airport_day") / F.col("avg_morning_delay_rate_airport"))
    )
  )

  df = df.drop("dep_hour", "is_early_morning")

  return df

In [0]:
df_baseline_filtered = add_airport_delay_windows(df_baseline_filtered)

In [0]:
df_baseline_filtered.printSchema()

In [0]:
df_baseline_filtered.count()


## Graph features

In [0]:
df_airport_codes = spark.read.csv('dbfs:/mnt/mids-w261/airport-codes_csv.csv', header=True, inferSchema=True)

df_airports_codes_clean = df_airport_codes.filter("iso_country='US'").filter("type !='closed'").filter("iata_code !='0'")

# Build airport by extracting lat/lon from airport codes
df_airports = df_airports_codes_clean \
    .filter(F.col("iata_code").isNotNull()) \
    .withColumn("lon", F.split("coordinates", ",").getItem(0).cast("double")) \
    .withColumn("lat", F.split("coordinates", ",").getItem(1).cast("double")) \
    .select("iata_code", "lat", "lon")

In [0]:
def haversine_distance(lat1_col, lon1_col, lat2_col, lon2_col, radius=6371):
    """
    Returns Haversine distance between two coordinate columns.
    
    Parameters
    ----------
    lat1_col, lon1_col : Column
        Latitude/Longitude of point A.
    lat2_col, lon2_col : Column
        Latitude/Longitude of point B.
    radius : float
        Earth radius in kilometers. Use 3959 for miles.
    """
    
    dlat = F.radians(lat2_col - lat1_col)
    dlon = F.radians(lon2_col - lon1_col)
    
    a = (
        F.pow(F.sin(dlat / 2), 2)
        + F.cos(F.radians(lat1_col))
        * F.cos(F.radians(lat2_col))
        * F.pow(F.sin(dlon / 2), 2)
    )
    
    c = 2 * F.asin(F.sqrt(a))
    
    return radius * c

In [0]:
# 1. Build base airport graph

def build_airport_graph(df_airport):
    a1 = df_airports.alias("a1")
    a2 = df_airports.alias("a2")

    df_graph = (
        a1.crossJoin(a2)
        .filter(F.col("a1.iata_code") != F.col("a2.iata_code"))
        .withColumn("distance_km",
                    haversine_distance(
                        F.col("a1.lon"), F.col("a1.lat"),
                        F.col("a2.lon"), F.col("a2.lat")
                    )
                )
    )

    # 2. Filter by distance (airport neighborhood definition)
    MAX_RADIUS_KM = 250  # recommended for airport delay propagation

    df_graph = df_graph.filter(F.col("distance_km") <= MAX_RADIUS_KM)

    # 3. Rank neighbors for each airport
    window = Window.partitionBy("a1.iata_code").orderBy("distance_km")

    df_graph = (
        df_graph
        .withColumn("neighbor_rank", F.row_number().over(window))
        .select(
            F.col("a1.iata_code").alias("airport_id"),
            F.col("a2.iata_code").alias("neighbor_airport_id"),
            "distance_km",
            "neighbor_rank",
            F.col("a2.lat").alias("neighbor_lat"),
            F.col("a2.lon").alias("neighbor_lon")
        )
    )

    return df_graph

In [0]:
df_graph = build_airport_graph(df_airports)

In [0]:
df_graph.printSchema()

In [0]:
# 2. Get airport-level delay signals
airport_level_delay_features = [
    'dep_delay_1h',
    'dep_delay_3h',
    'dep_delay_6h',
    'num_flights_3h',
    'max_dep_delay_3h'
]

df_airport_delay = df_baseline_filtered.select(
    "DEP_DEL15",
    "YEAR",
    "MONTH",
    "ORIGIN",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "sched_dep_ts",
    "dep_delay_1h",
    "dep_delay_3h",
    "dep_delay_6h",
    "num_flights_3h",
    "max_dep_delay_3h",
    "tail_avg_delay_10h",
    "tail_recent_delay",
    "route_avg_delay_6h",
    "route_max_delay_6h",
).withColumnRenamed("origin", "airport_id")

LOOKBACK_3H = 3600 * 3
CUTOFF_BEFORE_DEPARTURE = 3600 * 2

w_airport = (
    Window.partitionBy("airport_id")
        .orderBy(F.col("sched_dep_ts").cast("long"))
        .rangeBetween(-LOOKBACK_3H, CUTOFF_BEFORE_DEPARTURE)  # last 3 hours per airport
    )

df_airport_delay_latest = (
    df_airport_delay
    .withColumn("mean_delay_1h", F.avg("dep_delay_1h").over(w_airport))
    .withColumn("mean_delay_3h", F.avg("dep_delay_3h").over(w_airport))
    .withColumn("max_delay_recent", F.max("max_dep_delay_3h").over(w_airport))
    .withColumn("mean_num_flights_3h", F.avg("num_flights_3h").over(w_airport))
    .dropDuplicates(["airport_id"])   # keep latest row
)

In [0]:
df_airport_delay_latest = df_airport_delay_latest.persist(StorageLevel.MEMORY_AND_DISK)

In [0]:
df_airport_delay.count()

In [0]:
df_airport_delay_latest.count()

In [0]:
display(
    df_airport_delay_latest.limit(5)
)

In [0]:
display(
    df_graph.limit(5)
)

In [0]:
# 3. Join delay signals to the proximity graph
g = df_graph.alias("g")

d1 = df_airport_delay_latest.alias("src")
d2 = df_airport_delay_latest.alias("dst")

df_graph_enriched = (
    g
    .join(d1, F.col("g.airport_id") == F.col("src.airport_id"), "left")
    .join(d2, F.col("g.neighbor_airport_id") == F.col("dst.airport_id"), "left")
    .drop(F.col("src.airport_id"))
    .drop(F.col("dst.airport_id"))
)

In [0]:
df_graph_weighted = (
    df_graph_enriched
    .withColumn("distance_weight", 1 / (1 + F.col("distance_km")))
    .withColumn("delay_similarity",
                1 / (1 + F.abs(F.col("src.mean_delay_1h") - F.col("dst.mean_delay_1h"))))
    .withColumn("traffic_weight",
                (F.col("src.mean_num_flights_3h") + F.col("dst.mean_num_flights_3h")) / 2)
    .withColumn(
        "edge_weight",
        F.col("distance_weight") * 0.4 +
        F.col("delay_similarity") * 0.4 +
        F.col("traffic_weight")  * 0.2
    )
    .select(
        "airport_id",
        "neighbor_airport_id",
        "distance_km",
        "edge_weight",
        "neighbor_rank"
    )
)

In [0]:
df_neighbor_delay = (
    df_graph_weighted.alias("g")
    .join(df_airport_delay_latest.alias("d"),
          F.col("g.neighbor_airport_id") == F.col("d.airport_id"))
    .groupBy("g.airport_id")
    .agg(
        F.sum(F.col("edge_weight") * F.col("d.mean_delay_1h")).alias("weighted_neighbor_delay_1h"),
        F.sum(F.col("edge_weight") * F.col("d.mean_delay_3h")).alias("weighted_neighbor_delay_3h"),
        F.sum(F.col("edge_weight") * F.col("d.max_delay_recent")).alias("weighted_neighbor_max_delay"),
    )
)

In [0]:
df_vertices = (
    df_airports
    .filter("iata_code IS NOT NULL")
    .select(
        F.col("iata_code").alias("id")
    )
)

df_edges = (
    df_graph_weighted
    .select(
        F.col("airport_id").alias("src"),
        F.col("neighbor_airport_id").alias("dst"),
        "distance_km",
        "edge_weight",
        "neighbor_rank"
    )
)

df_edges = df_edges.filter("src IS NOT NULL AND dst IS NOT NULL")

g = GraphFrame(df_vertices, df_edges)


In [0]:
# Get total incoming and outgoing degrees for each airport
df_in_degree = g.inDegrees.withColumnRenamed("inDegree", "deg_in")
df_out_degree = g.outDegrees.withColumnRenamed("outDegree", "deg_out")

df_degree = df_in_degree.join(df_out_degree, "id", "outer")

# df_closeness = g.closeness(vertices="id").select("id", "closeness")
# df_eigen = g.eigenvector(vertices="id").select("id","eigenvector")

# Get rank value for each airport
df_pagerank = g.pageRank(resetProbability=0.15, maxIter=10).vertices \
    .select("id", F.col("pagerank").alias("graph_pagerank"))

df_hub_authority = df_degree.join(df_pagerank, "id", "left") \
    .withColumn("hub_score", F.col("deg_out") * F.col("graph_pagerank")) \
    .withColumn("authority_score", F.col("deg_in") * F.col("graph_pagerank"))

# df_clustering = g.clustering(vertices="id").select("id", "clustering").alias("graph_clustering")

# top_landmarks = [row['id'] for row in df_degree.orderBy(F.col("total_degree").desc()).limit(6).collect()]
# print(top_landmarks)

landmarks = ["JFK", "ATL", "ORD", "LAX", "DFW", "DEN"] # major airports acting like hub
df_betweenness = (
    g.shortestPaths(landmarks)
     .select("id", "distances")
     .withColumn("graph_betweenness",
                 F.size("distances"))
     .select("id", "graph_betweenness")
)

df_graph_features = df_hub_authority.join(df_betweenness, "id", "left") \
    .withColumnRenamed("id", "airport_id") \
    .fillna(0)

In [0]:
# checkpoint data
df_graph_features.write.mode("overwrite").parquet(
            f"{folder_path}/mvp/graph_data")

In [0]:
df_graph_features.count()


##Combine features

In [0]:
display(df_graph_features.limit(5))

In [0]:
df_airport_delay_latest.printSchema()

In [0]:
df_airport_delay.printSchema()

In [0]:
df_airport_delay.count()

In [0]:
df_graph_features.count()

In [0]:
df_features = (
    df_airport_delay
    .join(df_graph_features, "airport_id", "left")
)

In [0]:
df_features.count()

In [0]:
df_features.printSchema()

In [0]:
select_final_cols = ["DEP_DEL15","CRS_DEP_TIME","CRS_ARR_TIME", "YEAR", "MONTH",
                     "tail_avg_delay_10h","tail_recent_delay","route_avg_delay_6h", "route_max_delay_6h",
                     "dep_delay_1h", "dep_delay_3h", "dep_delay_6h", "num_flights_3h",
                     "deg_in", "deg_out", "graph_pagerank", "hub_score", "authority_score", "graph_betweenness"]

In [0]:
df_final = df_features.select(select_final_cols)

In [0]:
# Validate that the dataset contains data from 5 years
df_final.groupBy("YEAR") \
    .agg(F.count("*").alias("num_records")) \
    .orderBy("YEAR") \
    .show()

In [0]:
df_final.groupBy("DEP_DEL15") \
        .count() \
        .show()

In [0]:
df_final_checkpoint = df_final

In [0]:
# checkpoint data
df_final_checkpoint.write.mode("overwrite").partitionBy("YEAR", "MONTH").parquet(
            f"{folder_path}/mvp/raw_feature_data")

In [0]:
df_final = spark.read.parquet(f"{folder_path}/mvp/raw_feature_data")

In [0]:
display(df_final.limit(10))


## Test and Clean / Cast Data

In [0]:
df_final = df_final_checkpoint

In [0]:
df_final.count()

In [0]:
df_final = df_final.filter(col("DEP_DEL15").isNotNull())
df_final.groupBy("DEP_DEL15") \
        .count() \
        .show()

In [0]:
df_final.count()

In [0]:
# Remove duplicate rows
df_final = df_final.dropDuplicates()

In [0]:
df_final.count()

In [0]:
display(df_final.limit(5))

In [0]:
df_final.printSchema()

In [0]:
# get null count by column
null_counts = df_final.select(
    *[count(when(col(c).isNull(), 1)).alias(c) for c in select_final_cols]
).toPandas().T.reset_index()
null_counts.columns = ["col", "null_count"]
total = df_final.count()
null_counts['null_pct'] = (null_counts.null_count / total * 100).round(1)
display(null_counts)

In [0]:
# cast data to numeric
for c in select_final_cols:
    if c in df_final.columns and c != "YEAR" and c != "MONTH":
        df_final = df_final.withColumn(c, expr(f"try_cast(`{c}` as double)"))
    elif c == "YEAR" or c == "MONTH":
        df_final = df_final.withColumn(c, expr(f"try_cast(`{c}` as integer)"))


In [0]:
df_final.printSchema()

In [0]:
display(null_counts)

In [0]:
df_final.select(
    F.min("CRS_DEP_TIME").alias("min_time"),
    F.max("CRS_DEP_TIME").alias("max_time")
).show()

In [0]:
df_final.printSchema()


### Impute Data

In [0]:
# impute zeros into missing values in df_final
df_final = df_final.fillna(0)
df_final.printSchema()

In [0]:
numeric_cols = [c for c in df_final.columns if c != "DEP_DEL15" and dict(df_final.dtypes)[c] != 'string']
numeric_cols

## Model Readiness

In [0]:
df_final.count()

In [0]:
# Undersample the majority class for DEP_DEL15 = 0 for the years of 2015 - 2018 as train data (2015-2017 train with 2018 unweighted pure val before retraining on undersampled 2018)
df_train = df_final.filter(df_final["YEAR"].isin([2015, 2016, 2017, 2018]))

In [0]:
# Testing
display(df_train, limit = 5)

In [0]:

# Assuming 'df' and binary column 'label' (1=minority, 0=majority)
major_df = df_train.filter(col("DEP_DEL15") == 0)
minor_df = df_train.filter(col("DEP_DEL15") == 1)

# Calculate ratio
ratio = float(minor_df.count()) / float(major_df.count())
print("ratio = ", ratio)

# Sample majority
sampled_major_df = major_df.sample(withReplacement=False, fraction=ratio, seed=42)

# Combine datasets
undersampled_df = sampled_major_df.union(minor_df)

# Check results
undersampled_df.groupBy("DEP_DEL15").count().show()


In [0]:
df_train = undersampled_df

In [0]:
df_test = df_final.filter(df_final["YEAR"] == 2019)

In [0]:
# Normalization - Train
df_corr_train = df_train.select(*(numeric_cols+["DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_train = assembler.transform(df_corr_train).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_train = scaler.fit(df_vec_train)
df_scaled_mvp_train = scaler_model_train.transform(df_vec_train)

df_scaled_mvp_train = df_scaled_mvp_train.select("YEAR", "MONTH", "scaled_features", "label")
df_scaled_mvp_train = df_scaled_mvp_train.withColumnRenamed("scaled_features", "features")

In [0]:
# Normalization - Test
df_corr_test = df_test.select(*(numeric_cols+["DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_test = assembler.transform(df_corr_test).withColumn("label", col("DEP_DEL15").cast("double"))
df_scaled_mvp_test = scaler_model_train.transform(df_vec_test)


df_scaled_mvp_test = df_scaled_mvp_test.select("YEAR", "MONTH", "scaled_features", "label")
df_scaled_mvp_test = df_scaled_mvp_test.withColumnRenamed("scaled_features", "features")


## Move into Train/Val/Test Directories

In [0]:
dbutils.fs.rm(f"{folder_path}/mvp/data_separated", True)

In [0]:
# within the folder_path directory, need to create a dummy_separation directory, which contains train, val, test directories, each with year and month partitions

def create_separation(folder_path):
    for dir in ["train"]:
        for year in range(2015, 2018):
            for month in range(1, 13):
                dbutils.fs.mkdirs(f"{folder_path}/mvp/data_separated/{dir}/YEAR={year}/MONTH={month}")
    for dir in ["val"]:
        for year in range(2018, 2019):
            for month in range(1, 13):
                dbutils.fs.mkdirs(f"{folder_path}/mvp/data_separated/{dir}/YEAR={year}/MONTH={month}")
    for dir in ["test"]:
        for year in range(2019, 2020):
            for month in range(1, 13):
                dbutils.fs.mkdirs(f"{folder_path}/mvp/data_separated/{dir}/YEAR={year}/MONTH={month}")
create_separation(folder_path)
# df_base.write.mode("overwrite").partitionBy("year", "month").parquet(f"{folder_path}/dummy_separation/train/2015/1")

In [0]:
#2015-2018
df_scaled_mvp_train.write.mode("overwrite").partitionBy("YEAR", "MONTH").parquet(f"{folder_path}/mvp/data_separated/train")
print("Completed train")

#2019
df_scaled_mvp_test.write.mode("overwrite").partitionBy("YEAR", "MONTH").parquet(f"{folder_path}/mvp/data_separated/test")
print("Completed test")


In [0]:
display(df_scaled_mvp_test.limit(5))

In [0]:
display(df_scaled_mvp_train.limit(5))